In [ ]:
import sys; sys.path.append("..")
import os
import pandas as pd
from src.config import ESTACIONES, UTM_ESTACIONES, VARIABLES_METEO
from src.meteo import utm_a_latlon, actualizar_meteo, agregar_derivadas

%load_ext autoreload
%autoreload 2

In [ ]:
ARCHIVO = "../data/meteo_gran_concepcion.parquet"
CACHE = "../data/meteo_cache"

os.makedirs(CACHE, exist_ok=True)
coords = utm_a_latlon(UTM_ESTACIONES)
meteo = actualizar_meteo(ESTACIONES, coords, ARCHIVO, carpeta_cache=CACHE)

In [ ]:
for v in VARIABLES_METEO:
    print(f"  {v:28} {meteo[v].notna().mean()*100:5.1f}%")

In [ ]:
meteo = agregar_derivadas(meteo)
print("Proxy estancamiento activo:", f"{meteo['proxy_estancamiento'].mean()*100:.1f}% de las horas")

In [ ]:
contam = pd.read_parquet("../data/dataset_limpio.parquet")
contam["timestamp"] = pd.to_datetime(contam["timestamp"]).dt.floor("h")
meteo["timestamp"] = pd.to_datetime(meteo["timestamp"]).dt.floor("h")

cols = ["timestamp", "estacion"] + VARIABLES_METEO + ["viento_u", "viento_v", "proxy_estancamiento", "proxy_inversion"]
maestro = contam.merge(meteo[cols], on=["timestamp", "estacion"], how="left")

print(f"Maestro: {len(maestro):,} filas")
print(f"Con meteo: {maestro['boundary_layer_height'].notna().mean()*100:.1f}%")

In [ ]:
vars_corr = ["boundary_layer_height","wind_speed_10m","temperature_2m","precipitation","relative_humidity_2m"]
print(maestro[["mp2.5"] + vars_corr].corr()["mp2.5"].drop("mp2.5").sort_values().round(3))

In [ ]:
maestro.to_parquet("../data/dataset_maestro.parquet", index=False)
print("dataset_maestro.parquet ✅")